## Imports and setup

This notebook is in a customized project and requires a custumized python module (.py file) to run. By default the path to the start module for a custom project is ./src/"project_name"; it needs to contain a startup function. The default name of that function is "Run_process". In this project, [AI4SH](https://ai4soilhealth.eu), the path to the custom project is ./src/ai4sh and the first cell calls the Run_process function in that package.

In [ ]:
# Standard library imports
from os import path, getcwd

import sys 

# Get the working directory of the notebook
notebook_path = getcwd()

# Set the path to the root directory of this jupyter notebook (python) project
module_path = path.abspath(path.join('../..'))

if module_path not in sys.path:
    
    sys.path.append(module_path)

# Package application imports
from src.lib import Initiate_process

# Import the custom class for running processes defined in the ai4sh package
from src.ai4sh import Run_process

## Scheme file
To run this cell you need to set the [scheme_file](https://xspatula.github.io/setup_core_db_docs/framework/scheme_file/), that defines the location (path) to the project and associated process files and your user. You can set the path to the scheme file with a relative path (relative to this notebook), in your home directory using tilde (~) or with an absolute path in the syntax of your operating system. 

In [ ]:
# Set the path to the scheme_file  

scheme_file = '../scheme_ai4sh.json'

# 0 Introduction
The preprocesses in this notebook need a dataframe with both spectra and indicators — a compiled dataset (see `compile_dataframe.ipynb`), usually after an unsupervised chain (e.g. `.snv.d1.mc`).

- The bands are selected on the **training samples** only: the test samples are the same holdout (hash of the sample id, `test_size`) as in `regression_modeling`, so the train/test metrics stay honest.
- Covariates of the compiled dataset (e.g. soil texture) are never selected and never treated as indicators; they are carried to the output.
- Each selection is saved as a new dataframe with a chain entry. `regression_modeling` replays it on its own training samples: univariate selection (one indicator per selection) is refitted, the other selections keep their bands (k-fold metrics then still include the selection, the note says so).
- `n_features` sets the number of bands to keep; 0 = the number RFECV chose, else you are asked after the plot.

# 1 Supervised band selection

## 1.1 Univariate selection

### 1.1.1 Univariate selection with SelectKBest
The `n_features` bands with the strongest univariate (F-test) relation to each indicator (SelectKBest). Fast and model-free, but blind to bands that only matter together.

suffix: `.uv-<indicator>` (separate selections) or `.uv` (all indicators in one selection)

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess_compiled/univariate_selection_c-org+n-tot_k30.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 1.2 Permutation, RFE and tree-based importance
One selection per indicator × regressor × selector:

- **permutation**: the drop in r² when a band is shuffled, out-of-fold on the training samples (a model fitted on the other folds is permuted on each fold).
- **rfe**: recursive feature elimination — the regressor is refitted while the weakest bands (coefficients or impurity importance) are removed one by one (or a fraction per round, `step` < 1). knn, mlp and svr (rbf) have neither, permutation is used instead.
- **tree_based**: impurity importance of an extra-trees ensemble (no regressor needed).

suffix: `.<indicator>-<regressor>-pms`, `.<indicator>-<regressor>-rfe` or `.<indicator>-tree`

### 1.2.1 Permutation, RFE and tree-based importance (fixed number of bands)

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess_compiled/band_selection_permutation+rfe+tree_c-org_plsr_k30.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 1.2.2 RFE with cross-validation (RFECV)
With `"rfe": {"CV": 5}` in the selector settings (`spectral_band_selector/selector_rfecv5.json`) RFE scores every number of bands by 5-fold cross-validation on the training samples and keeps the best; with `n_features` 0 that number is saved without asking. The name extension `#cv5` keeps these files apart from the fixed-size selections.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess_compiled/band_selection_rfecv_c-org_plsr+randfor.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

# 2 Wavelets
The discrete wavelet transform (DWT) decomposes each spectrum into an approximation (the broad shape) and detail coefficients per scale (local, sharp features). It is unsupervised (no indicators needed), so the transform itself is a step of the spectral pre-processing (`spectra_wavelet` in `ml_preprocess_separated.ipynb`, or the `wavelet` item of a chemometric chain):

- `"mode": "denoise"` — the detail coefficients are soft-thresholded and the spectrum reconstructed: the same bands, less noise (suffix `.wd`).
- `"mode": "coefficients"` — the approximation coefficients at `level` (plus `details` detail levels, coarsest first) replace the bands as features `wt1 .. wtn` (suffix `.wt`).

The band selectors above then select coefficients instead of bands.

## 2.1 Wavelet coefficients 

### 2.1.1 Wavelet coefficients in a chemometric chain

The chain asks `Accept and save snv.d1.mc.wt result? [y/n]` — answer `y`: the next cells need the saved `.snv.d1.mc.wt` dataframe.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess_compiled/chemometric_chain_snv-d1-mc-wavelet-coefficients.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 2.2 Selection of wavelet coefficients

### 2.2.1 Select wavelet coefficients

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess_compiled/band_selection_rfe_c-org_plsr_wavelet-coefficients_k20.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

# 3 Model the selections


## 3.1 Compare the selections


### 3.1.1 Train/test models on all bands and on each selection
`regression_modeling` on the full pre-processed spectra and on each selection (train/test), to compare them. The process file holds one regression block per dataframe; run the selection cells above first (a missing dataframe is reported and skipped).

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_preprocess_compiled/regression_modeling_c-org_band-selections_tt.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)